In [ ]:
#Importation des bibliothèques requises pour le traitement des données.
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy import stats
from fanalysis.pca import PCA as fPCA
from fanalysis.mca import MCA as fMCA
from fanalysis.ca import CA as fCA

%matplotlib inline

# Import et traitement du fichier : "population-annuelle-communes-metropole-lyon.csv"

In [ ]:
#Récupérer la table "population-annuelle-communes-metropole-lyon".
csv_population = pd.read_csv("../data/population-annuelle-communes-metropole-lyon.csv", sep=";")
#Le fichier quand chargé dans l'IDE était illisible, le problème était le fait que la séparation était faite avec le caractère ";" au lieu de la virgule reconnue automatiquement par panda. Nécessité de changer le caractère dans le code.

#Choix de garder uniquement ces 3 colonnes expliqué directement dans le dossier.
csv_population = csv_population[['insee','nom', 'pop2022']]

#Transformation de la colonne "insee" en chaîne caratères afin d'éviter des problèmes dans le traitement des attributs.
csv_population = csv_population.set_index('insee')

#Code afin de se débarasser de la ligne 'Lyon' dans le CSV.
csv_population = csv_population[csv_population["nom"] != "Lyon"]

#Code afin d'enlever les valeurs manquantes, ces lignes correspondent notamment à la population de Lyon distribuée par arrondissement dans le CSV.
csv_population=csv_population.dropna()
csv_population.head()

#Import et traitement du fichier : "equipements-publics-metropole-lyon.csv"

In [ ]:
csv_equipement = pd.read_csv("../data/equipements-publics-metropole-lyon.csv",  sep=";")
csv_equipement2 = csv_equipement[["uid", "theme", "type", "nom", "adresse", "codepostal", "commune", "insee"]]
csv_equipement2.set_index("uid")
csv_equipement2.head()
csv_equipement2 = csv_equipement2.dropna()
csv_equipement2

#Ici, nous avons mis le code "sep=;" car dans le csv, ce sont des points-virgules qui séparent les différentes valeurs
#Également, nous avons ajouté le code ".dropna()" pour se débarrasser des valeurs manquantes afin de nous faciliter le travail de code et d’analyse mais aussi car sinon notre analyse de composantes pourrait ne pas être efficace si on garde même les valeurs manquantes (le résultat serait faussé)
#Nous avons choisi de mettre uid en index car ce dernier est un identifiant unique de chaque enregistrement.



---



# Jointure des deux fichiers

In [ ]:
#Avec ce code nous procédons à la jointure des deux tables avec "insee" comme clé commune. La méthode "inner" est choisie afin de préserver les attributs des deux tables.
csv_commun = pd.merge(
    csv_population,
    csv_equipement2,
    on="insee",
    how='inner')

#Une fois la jointure effectuée, c'est possible d'améliorer le nouveau dataframe en retirant les colonnes redondantes et en recodant les titres peu parlants.
csv_commun = csv_commun.drop(columns=['nom_x', 'uid', 'adresse', 'codepostal'])
csv_commun.rename(columns={'nom_y': 'nom_équipement', 'pop2022' : 'population_2022' }, inplace=True)

#On met l'attribut "insee" comme index du dataframe
csv_commun = csv_commun.set_index('insee')

#Vérifiaction des effectifs de l'attribut "theme"
print(csv_commun["theme"].value_counts())
#Certains doublons avec des valeurs comme "Equipement  Culturel" et "Equipement culturel". Pour pandas il s'agit de deux valeurs différentes alors qu'il s'agit de la même chose.

csv_commun

In [ ]:
#Il faut donc recoder les valeurs en harmonisant l'écriture et en regroupant les catégories. On effectue ceci avec un objet de type dictionnaire.
correction={"Equipement  Culturel":"Equipement culturel", "Equipement  culturel":"Equipement culturel",
            "Equipement Cultuel" : "Equipement cultuel",
            "Autre service à la population" : "Autre équipement", "Autres équipements publics" : "Autre équipement",
            "Equipement  social et d'animation" : "Equipement social et d'animation", "Equipement social et de la Solidarité" : "Equipement social et d'animation",
            "Equipement d'enseignement " : "Equipement d'enseignement", "Equipement d'enseignement professionnel" : "Equipement d'enseignement",
            "Equipement de Santé" : "Equipement sanitaire"
            }

#Cette ligne nous permet d'appliquer la correction des modalités de l'attribut "theme"
csv_commun=csv_commun.replace(correction)
print(csv_commun["theme"].value_counts())

# Représentation des données : effectifs des équipements par thème.

In [ ]:
#Construction de différents graphiques pour représenter les effectifs des équipements par thème.
distribution1 = csv_commun["theme"].value_counts()
plt.figure()
distribution1.plot(kind="barh", title="Effectifs des équipements par thème")
plt.ylabel("Thème de l'équipement")
plt.show()

#Avec les effectifs normalisés.
distribution2 = csv_commun["theme"].value_counts(normalize=True)
plt.figure()
distribution2.plot(kind="barh", title="Proportion des équipements par thème")
plt.ylabel("Thème de l'équipement")
plt.xlabel("Pourcentage %")
plt.show()

# Représentation des données : Nombre d'équipements par commune.

In [ ]:
#Cette ligne de code stocke dans une nouvelle variables les effectifs des équipements (tout theme confondu)
#Changement de l'index du dataframe afin d'avoir les communes en ligne.
equipements_commune = (
    csv_commun
    .groupby('commune')['theme']
    .count()
    .reset_index()
)

#Avec ce code le comptage des équipements deviennent une nouvelle colonne nommée "nb_equipements".
equipements_commune.columns = ['commune', 'nb_equipements']
equipements_commune
equipements_commune=equipements_commune.set_index('commune')

#Construction d'un graphique pour représenter le nombre d'équipements par commune.
plt.figure()
equipements_commune.plot(kind="barh", title="Effectifs des équipements par commune")
plt.ylabel("Commune")
plt.show()


# Méthode factorielle exploratoire (ACM) :  Préparation

Notre problématique cherche à croiser
d'un côté : le thème de l'équipement (variable qualitative), et
de l'autre la population (variable quantitative). Nous ne pouvons pas croiser ces deux types de données sans faire d'autres traitements. Nous avons fait le choix de coder la population en classe artificielles (Petite, Moyenne, Grande) pour transformer la population en variable qualitative. Le choix c'est fait prenant des intervalles qui puissent représenter toutes les communes présentes. Ceci nous permettra d'effectuer une ACM qui croisera les différents types d'équipements avec la population recodée en variable qualitative.


In [ ]:
# Cette ligne construit un tableau de contingence : communes en lignes, thèmes en colonnes.
# Ayant comme colonne les différents thèmes des équipements.
tableau_contingence = pd.crosstab(
    csv_commun["commune"].reset_index(drop=True),  # reset_index : l'index "insee" n'est pas unique (pandas 3)
    csv_commun["theme"].reset_index(drop=True)
)

tableau_contingence

# Cette ligne permet de regrouper les communes par la valeur de l'attibut "population_2022". Créant la colonne "population_2022" dans le tableau.
# ".first()" permet d'éviter la répétition de la valeur de l'attibut "population_2022" présent dans le tableau original avec les équipements en ligne.
tableau_contingence["population_2022"] = (
    csv_commun
    .groupby("commune")["population_2022"].first()
)

# Ce code nous permet de vérifier la population de chaque commune pour établir un classement appropié.
print(tableau_contingence["population_2022"].sort_values())

A partir de ces informations on peux faire un classement adapté des communes.

On peux observer que la population oscillent entre 6000 et 52000 habitants. Afin d'avoir une typologie adaptée il faudra choisir des bornes qui distribuent équitablement les communes :
- Saint-Cyr-au-Mont-d'Or, Champagne-au-Mont-d'Or, Saint-Didier-au-Mont-d'Or, et Neuville-sur-Saône ont moins de 8000 habitants.
- Écully, Givors et Rillieux-la-Pape ont entre 15 000 et 35 000 habitants.
- Uniquement Vaulx-en-Vellin a plus de 50 000 habitants.

Nous pouvons construire trois catégories ("Petite", "Moyenne", "Grande")

Vaulx-en-Velin se distingue des autres communes avec 52 448 habitants. Malgré le fait que'elle est seule dans la catégorie 'Grande', nous avons choisi de la conserver car l'écarter réduirait encore plus les individus statistiques déjà très bas pour l'exercice.


In [ ]:
# Cette ligne permet d'assigner à chaque commune une étiquette adaptée à sa population pour le codage en variable qualitative.
tableau_contingence["Taille commune"] = pd.cut(
    tableau_contingence["population_2022"],
    bins=[0, 8000, 35000, 60000],
    labels=["Petite", "Moyenne", "Grande"]
)

# Cette ligne retire la population brute, afin de préserver uniquement la version qualitative.
tableau_contingence = tableau_contingence.drop(columns=["population_2022"])

Visualisation préliminaire des communes par thème d'équipement

In [ ]:
# Cette ligne crée un graphique qui représente par commune l'effectifs des thèmes des équipements. Il s'agit d'un graphique imparfait qui pourrait être amélioré.
tableau_contingence.plot.bar(figsize=(10,10), title="Effectifs des thèmes d'équipements par commune")
# Ceci nous permet d'observer que Vaulx-en-Vellin concentre un nombre d'équipements fortement supérieur à celui des autres communes.
# Nous essayerons de confirmer si la population a une incidence sur ces résultats grâce à l'ACM.

# Méthode factorielle exploratoire (ACM) : Création des matrices

In [ ]:
# Cette ligne nous permet de créer la matrice R.
R = pd.DataFrame(tableau_contingence, columns = ["Autre équipement", "Equipement administratif", "Equipement cultuel", "Equipement culturel", "Equipement d'enseignement", "Equipement d'enseignement privé", "Equipement de justice", "Equipement de mobilité", "Equipement sanitaire", "Equipement social et d'animation", 	"Equipement sociaux et de la Petite enfance", "Equipement sportif et de loisirs", "Taille commune"]) #Pour reprendre les notations du cours
R

# Cette ligne nous permet de créer la matrice indicatrice Z.
Z=pd.get_dummies(R,prefix='',prefix_sep='')
Z

Nous retrouvons un problème, la matrice Z fonctionne avec le nouveau codage des populations car il s'agit d'une variable qualitative. Cependant les effectifs des équipements ne sont pas qualitatifs. C'est possible de répondre à ce problème en plafonnant les valeurs des effectifs à 1 avec "pandas.DataFrame.clip", réflettant l'abscence ou la préscence d'un type d'équipement dans la commune respective. En faisant ceci le nombre d'équipements par commune n'aura pas d'influence, laissant de côté une partie de l'analyse. Pour répondre à cette limite nous ferons un traitement ultérieurement pour prendre en compte les effectifs des équipements par commune.  

**Par ailleurs il y a des thèmes d'équipements qui sont présents dans toutes les communes : 'Equipement culturel', "Equipement d'enseignement", "Equipement social et d'animation", 'Equipement sportif et de loisirs'. Ceci nous permet de supposer qu'il s'agit d'équipements essentiels étant donnée qu'ils sont présents dans toutes les communes.** Malgré l'importance de cette information il faut traiter cette particularité afin de réaliser l'ACM.

Afin de répondre à cette difficulté nous pouvons créer un nouveau dataframe avec uniquement les équipements secondaires. Il faut noter que également que le thème "Equipement d'enseignement privé" est manquant dans l'ensemble de communes donc il faudra également l'enlever du dataframe. Ceci est parlant quant à la préscence des équipements d'enseignements privés dans les communes présentes dans l'analyse.  

Nous ferons d'abord les modifications avec "pandas.DataFrame.clip" pous modifier la population. Puis nous réconstruirons le deuxième dataframe avec les thèmes secondaires.

In [ ]:
# Avec cette ligne on crée un deuxième tableau de contingence pour effectuer les modifications.
tableau_contingence2 = tableau_contingence

# Ce code nous permettra de stocker les colonnes dans une variable qu'on modifieras ensuite avec la fonction "pandas.DataFrame.clip".
# Ce code nous permet également d'enlever la colonne "Taille commune", étant donnée qu'elle est déjà qualitative.
cols_themes = tableau_contingence2.columns.drop("Taille commune")
# Ce code modifie les valeurs pour les préparer au codage disjonctif avec "pandas.DataFrame.clip".
tableau_contingence2[cols_themes] = tableau_contingence2[cols_themes].clip(upper=1)

tableau_contingence2
#Nous pouvons ensuite de créer les matrices pour l'ACM.

R1 = pd.DataFrame(tableau_contingence2, columns = ["Autre équipement", "Equipement administratif", "Equipement cultuel", "Equipement de justice", "Equipement de mobilité", "Equipement sanitaire", "Equipement sociaux et de la Petite enfance", "Taille commune"])
R1.head()

# Cette ligne nous permet de créer la matrice indicatrice Z.
Z1=pd.get_dummies(R1,prefix='',prefix_sep='')
Z1.head()
# Les valeurs pour les colonnes respectives aux thèmes ne sont pas en booléen ce qui pourra poser problème lors de la réalisation de l'ACM.



# Méthode factorielle exploratoire (ACM) : Dimensions des matrices et valeurs propres

In [ ]:
# Ce code nous permet d'obtenir les dimensions de la matrice R.
n=R1.shape[0]
print("R1 : n=",n)
s=R1.shape[1]
print("R1 : s=",s)


# On obtient les dimensions de la matrice Z.
n=Z1.shape[0]
print("Z1: n=",n)
p=Z1.shape[1]
print("Z1: p=",p)

# On va faire le calcul pour obtenir le nombre de valeurs propres
nb_vp = p-s
print("Nombre max de valeurs propres = ",nb_vp)

# Méthode factorielle exploratoire (ACM) : Réalisation de l'ACM

In [ ]:
# Ce code passe les valeurs de la matrice indicatrice en entiers afin de pouvoir effectuer l'ACM. Sinon la différence entre booléens et entiers (0,1) ne permet pas le traitement
Z1 = Z1.astype(int)

mca1 = fMCA(row_labels=R1.index.values, var_labels=R1.columns.values)
mca1.fit(Z1.to_numpy())

mca1.plot_eigenvalues(type="percentage")

Les deux premiers axes représentent autour de 60% de la variance. Le critère du coude ne s'applique pas clairement. Il faut appliquer la correction de Benzécri, à partir des outils vu en cours dans le notebook "SDF".

In [ ]:
# Récupération des valeurs propres supérieures à (1/s).
lambada =mca1.eig_[0][mca1.eig_[0]>1/s]
print(lambada)

# Ce code applique la correction.
lambada_prim =((s/(s-1))*(lambada-1/s))**2
print(lambada_prim)

# Calcule de la nouvelle inertie
I_prim =np.sum(lambada_prim)
print(I_prim)

# Production des pourcentages.
percent_prim =lambada_prim/I_prim*100

# Transformation de la distribution en deux listes x pour les abscisses et y pour les ordonnées.
x = range(0,len(percent_prim))
y = list(percent_prim)

# On passe x et y pour créer un nouveau graphique.
plt.bar(x,y)
#On peux donc retentir les deux premiers axes.

# Méthode factorielle exploratoire (ACM) : Extraction et analyse des résultats

In [ ]:
# Ce code nous permet d'obtenir les coordonnées, la contribution et le cosinus² des points-lignes.
print(mca1.row_coord_)
print(mca1.row_contrib_)
print(mca1.row_cos2_)

# Ce code nous permet d'obtenir les coordonnées, la contribution et le cosinus² des points-colonnes.
print(mca1.col_coord_)
print(mca1.col_contrib_)
print(mca1.col_cos2_)

# Afin d'avoir une meilleur lisibilité nous pouvons l'ensemble de ces données à un dataframe.
bilan_lignes = mca1.row_topandas()
bilan_colonnes = mca1.col_topandas()

bilan_colonnes



Malheureusement lors du traitement les "labels" Des points colonnes sont notés automatiquement avec un codage "col0_0", "col0_1", etc... Je n'ai pas réussi à modifier ceci ce qui rend le résultat pour les points colonnes illisible. En revanche c'est possible de tracer les graphique des contributions pour les point lignes.


# Méthode factorielle exploratoire (ACM) : Contributions et représentations pour les points-lignes.

In [ ]:
# Ce code crée un graphique qui représente la contribution de chaque observation à l'inertie de l'axe 1.
mca1.plot_row_contrib(num_axis=1)

# Ce code crée un graphique qui représente la contribution de chaque observation à l'inertie de l'axe 2.
mca1.plot_row_contrib(num_axis=2)

Individus bien représentés dans l'axe 1 et l'axe 2.

In [ ]:
# Ce code crée un graphique qui montre les individus bien représentés par l'axe 1.
mca1.plot_row_cos2(num_axis=1)

# Ce code crée un graphique qui montre les individus bien représentés par l'axe 2.
mca1.plot_row_cos2(num_axis=2)

Graphique des observations projetées sur les deux premiers axes.

In [ ]:
mca1.mapping_row(num_x_axis=1, num_y_axis=2, figsize = (7,7))

# Méthode d'Analyse des Composantes Principales (ou ACP) :  Préparation du tableau

Ayant effectué l'ACM pour les variables qualitatives, nous pouvons nous concentrer sur les variables quantitatives. Nous pouvons ainsi reprendre le tableau original avant la réalisatio de l'ACM afin de croiser le nombre d'équipements avec la population.

In [ ]:
#création du tableau des valeurs de l'ACP
tableau_acp = pd.crosstab(
    csv_commun["commune"].reset_index(drop=True),  # reset_index : l'index "insee" n'est pas unique (pandas 3)
    csv_commun["theme"].reset_index(drop=True)
)

# Ajouter la population par commune
population_commune = csv_commun[["commune", "population_2022"]]
population_commune = population_commune.set_index("commune")

tableau_acp["population_2022"] = (
    csv_commun
    .groupby("commune")["population_2022"].first())
tableau_acp
#ce code permet de regrouper les villes ensembles ("groupby") et également de ne prendre que la 1ère valeur de chaque groupe ("first")

# Méthode d'Analyse des Composantes Principales (ou ACP) :  Réalisation de l'ACP

In [ ]:
pca= fPCA(std_unit=True, row_labels=tableau_acp.index, col_labels=tableau_acp.columns)
pca.fit(tableau_acp.to_numpy())

In [ ]:
print(pca.eig_)
# Code qui permet de réaliser un graphique des valeurs propres pour la séléction des nombres d'axes.
pca.plot_eigenvalues()

In [ ]:
# Ce code permet de réaliser le même graphique mais avec les valeurs propres en % d'inertie
pca.plot_eigenvalues(type="percentage")

Ensuite il s'agit de projeter l'ensemble des observations sur les deux plans factoriels.

In [ ]:
# Code qui permet de réaliser le graphique des individus sur l'espace factoriel.
pca.mapping_row(num_x_axis=1, num_y_axis=2,figsize=(10,10))